
# 26 — JCIM Manuscript Figure Generation — Stable Plotting Version

This version is adapted to the existing project execution pattern.

**Important:** Matplotlib is **not imported into the Jupyter kernel**.  
Instead, all plotting runs in the same separate stable Anaconda Python process used by the existing manuscript notebook.

Output:

```text
figures/jcim_manuscript_pdf/
```

All figures are saved as PDF with:

```python
dpi=500
bbox_inches="tight"
```


In [1]:

from pathlib import Path
import os
import subprocess
import tempfile

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError(
        "Project root not found. Run this notebook from the repository root "
        "or toxicity_screening_project/notebooks/."
    )

# Use the same stable plotting interpreter already used by notebook 25.
plot_python = Path(
    os.environ.get(
        "TOXICITY_PLOT_PYTHON",
        r"D:\Users\anaconda3\python.exe",
    )
)

if not plot_python.exists():
    raise FileNotFoundError(
        f"Stable plotting Python was not found: {plot_python}\n"
        "Set environment variable TOXICITY_PLOT_PYTHON to the Python executable "
        "used successfully by notebook 25."
    )

print({
    "root": str(ROOT),
    "plot_python": str(plot_python),
})


{'root': 'D:\\Dropbox\\Work\\Learning\\Python\\toxicity_screening_project', 'plot_python': 'D:\\Users\\anaconda3\\python.exe'}



## Generate all JCIM manuscript figures in a separate plotting process

If Matplotlib or another compiled plotting dependency crashes, only the child process terminates; the Jupyter kernel remains alive and the subprocess output is shown below.


In [2]:

plot_script = '\nfrom pathlib import Path\nimport warnings\nimport numpy as np\nimport pandas as pd\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\n\nROOT = Path.cwd().resolve()\nTABLES = ROOT / "tables"\nRESULTS = ROOT / "results"\nFIG_DIR = ROOT / "figures" / "jcim_manuscript_pdf"\nFIG_DIR.mkdir(parents=True, exist_ok=True)\n\nplt.rcParams.update({\n    "figure.dpi": 140,\n    "savefig.dpi": 500,\n    "font.size": 9,\n    "axes.titlesize": 10,\n    "axes.labelsize": 9,\n    "legend.fontsize": 8,\n    "xtick.labelsize": 8,\n    "ytick.labelsize": 8,\n    "pdf.fonttype": 42,\n    "ps.fonttype": 42,\n    "axes.spines.top": False,\n    "axes.spines.right": False,\n})\n\nENDPOINT_ORDER = [\n    "herg_blockade",\n    "ames_mutagenicity",\n    "SR-p53",\n    "SR-ATAD5",\n    "SR-ARE",\n    "SR-MMP",\n]\n\nENDPOINT_LABELS = {\n    "herg_blockade": "hERG",\n    "ames_mutagenicity": "Ames",\n    "SR-p53": "SR-p53",\n    "SR-ATAD5": "SR-ATAD5",\n    "SR-ARE": "SR-ARE",\n    "SR-MMP": "SR-MMP",\n}\n\nMODEL_LABELS = {\n    "dummy": "Dummy",\n    "logistic_regression": "Logistic regression",\n    "random_forest": "Random forest",\n    "svm": "SVM",\n    "gradient_boosting": "Gradient boosting",\n    "xgboost": "XGBoost",\n    "mlp": "MLP",\n    "fingerprint_mlp": "Fingerprint MLP",\n    "gnn": "GNN",\n    "multitask_fingerprint": "MT fingerprint",\n    "multitask_gnn": "MT GNN",\n}\n\ndef endpoint_label(value):\n    return ENDPOINT_LABELS.get(value, str(value))\n\ndef model_label(value):\n    if value in MODEL_LABELS:\n        return MODEL_LABELS[value]\n    return str(value).replace("_", " ").replace("-", " ").title()\n\ndef require_columns(df, required, source_name):\n    missing = [c for c in required if c not in df.columns]\n    if missing:\n        raise ValueError(\n            f"{source_name} missing columns: {missing}; "\n            f"available={list(df.columns)}"\n        )\n\ndef load_csv(path, required=None):\n    path = Path(path)\n    if not path.exists():\n        raise FileNotFoundError(f"Required file not found: {path}")\n    df = pd.read_csv(path)\n    if required:\n        require_columns(df, required, path.name)\n    return df\n\ndef optional_csv(path, required=None):\n    path = Path(path)\n    if not path.exists():\n        print(f"[SKIP] optional file not found: {path}")\n        return None\n    df = pd.read_csv(path)\n    if required:\n        require_columns(df, required, path.name)\n    return df\n\nmanifest = []\n\ndef save_pdf(fig, stem, description, source_files):\n    out = FIG_DIR / f"{stem}.pdf"\n    fig.savefig(out, format="pdf", dpi=500, bbox_inches="tight")\n    plt.close(fig)\n    manifest.append({\n        "figure": stem,\n        "file": str(out.relative_to(ROOT)),\n        "description": description,\n        "source_files": "; ".join(str(Path(p).relative_to(ROOT)) for p in source_files),\n        "dpi": 500,\n        "format": "PDF",\n    })\n    print(f"[PLOT] created {out}")\n    return out\n\n# ------------------------------------------------------------------\n# Figure 1 — Endpoint data landscape\n# ------------------------------------------------------------------\nendpoint_path = TABLES / "manuscript_endpoint_summary.csv"\nif not endpoint_path.exists():\n    endpoint_path = TABLES / "endpoint_summary.csv"\n\nendpoint_df = load_csv(\n    endpoint_path,\n    required=[\n        "endpoint", "observed_labels", "missing_labels",\n        "positive_prevalence", "unique_scaffolds"\n    ],\n).copy()\n\nendpoint_df["endpoint"] = pd.Categorical(\n    endpoint_df["endpoint"], categories=ENDPOINT_ORDER, ordered=True\n)\nendpoint_df = endpoint_df.sort_values("endpoint")\nendpoint_df["endpoint_label"] = endpoint_df["endpoint"].map(endpoint_label)\n\nfig, axes = plt.subplots(1, 2, figsize=(8.0, 3.8))\nx = np.arange(len(endpoint_df))\nwidth = 0.36\n\naxes[0].bar(x - width/2, endpoint_df["observed_labels"], width, label="Observed labels")\naxes[0].bar(x + width/2, endpoint_df["missing_labels"], width, label="Missing labels")\naxes[0].set_xticks(x)\naxes[0].set_xticklabels(endpoint_df["endpoint_label"], rotation=35, ha="right")\naxes[0].set_ylabel("Number of molecules")\naxes[0].set_title("A. Endpoint label availability")\naxes[0].legend(frameon=False)\n\naxes[1].bar(x, 100 * endpoint_df["positive_prevalence"])\naxes[1].set_xticks(x)\naxes[1].set_xticklabels(endpoint_df["endpoint_label"], rotation=35, ha="right")\naxes[1].set_ylabel("Positive prevalence (%)")\naxes[1].set_title("B. Positive-class prevalence")\n\nfor i, row in endpoint_df.reset_index(drop=True).iterrows():\n    axes[1].text(\n        i,\n        100 * row["positive_prevalence"],\n        f\'{100 * row["positive_prevalence"]:.1f}%\',\n        ha="center",\n        va="bottom",\n        fontsize=7,\n    )\n\nfig.tight_layout()\nsave_pdf(\n    fig,\n    "Fig01_endpoint_data_landscape",\n    "Observed/missing endpoint labels and positive-class prevalence.",\n    [endpoint_path],\n)\n\n# ------------------------------------------------------------------\n# Figure 2 + supplementary metric heatmaps\n# ------------------------------------------------------------------\nmodel_df = load_csv(\n    TABLES / "manuscript_model_comparison.csv",\n    required=["endpoint", "model", "partition", "pr_auc", "mcc", "brier"],\n).copy()\n\nif "split_strategy" in model_df.columns:\n    model_df = model_df.loc[model_df["split_strategy"].eq("global_scaffold")].copy()\n\nmodel_df = model_df.loc[model_df["partition"].eq("test")].copy()\nmodel_df = model_df.loc[~model_df["model"].eq("dummy")].copy()\n\ndef metric_heatmap(df, metric, title, stem, higher_is_better=True):\n    pivot = df.pivot_table(\n        index="endpoint",\n        columns="model",\n        values=metric,\n        aggfunc="max" if higher_is_better else "min",\n    )\n    pivot = pivot.reindex([e for e in ENDPOINT_ORDER if e in pivot.index])\n\n    score = pivot.mean(axis=0, skipna=True)\n    model_order = score.sort_values(ascending=not higher_is_better).index.tolist()\n    pivot = pivot[model_order]\n\n    fig, ax = plt.subplots(figsize=(max(7.0, 0.9 * len(model_order)), 4.3))\n    image = ax.imshow(pivot.to_numpy(dtype=float), aspect="auto")\n\n    ax.set_xticks(np.arange(len(model_order)))\n    ax.set_xticklabels([model_label(m) for m in model_order], rotation=40, ha="right")\n    ax.set_yticks(np.arange(len(pivot.index)))\n    ax.set_yticklabels([endpoint_label(e) for e in pivot.index])\n    ax.set_title(title)\n\n    for i in range(pivot.shape[0]):\n        for j in range(pivot.shape[1]):\n            value = pivot.iloc[i, j]\n            if pd.notna(value):\n                ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=7)\n\n    cbar = fig.colorbar(image, ax=ax, shrink=0.85)\n    cbar.set_label(metric.replace("_", " ").upper())\n    fig.tight_layout()\n\n    save_pdf(\n        fig,\n        stem,\n        f"{metric} across models on the primary global-scaffold test set.",\n        [TABLES / "manuscript_model_comparison.csv"],\n    )\n\nmetric_heatmap(\n    model_df, "pr_auc",\n    "Scaffold-aware test performance: PR-AUC",\n    "Fig02_scaffold_test_pr_auc", True\n)\n\nmetric_heatmap(\n    model_df, "mcc",\n    "Scaffold-aware test performance: MCC",\n    "FigS01_scaffold_test_mcc", True\n)\n\nmetric_heatmap(\n    model_df, "brier",\n    "Scaffold-aware test reliability: Brier score",\n    "FigS02_scaffold_test_brier", False\n)\n\n# ------------------------------------------------------------------\n# Figure 3 — Reliability trade-off\n# ------------------------------------------------------------------\ncal_df = load_csv(\n    TABLES / "manuscript_calibration.csv",\n    required=[\n        "endpoint", "selected_model", "selected_calibrator",\n        "test_pr_auc", "test_brier", "test_ece"\n    ],\n).copy()\n\nfig, ax = plt.subplots(figsize=(6.5, 4.6))\nax.scatter(cal_df["test_pr_auc"], cal_df["test_brier"], s=55)\n\nfor _, row in cal_df.iterrows():\n    ax.annotate(\n        f\'{endpoint_label(row["endpoint"])}\\nECE={row["test_ece"]:.3f}\',\n        (row["test_pr_auc"], row["test_brier"]),\n        xytext=(5, 5),\n        textcoords="offset points",\n        fontsize=7,\n    )\n\nax.set_xlabel("Scaffold-test PR-AUC")\nax.set_ylabel("Scaffold-test Brier score (lower is better)")\nax.set_title("Discrimination and probability reliability after calibration")\nax.grid(alpha=0.2)\nfig.tight_layout()\n\nsave_pdf(\n    fig,\n    "Fig03_reliability_tradeoff",\n    "Endpoint-level PR-AUC versus Brier score for selected calibrated models.",\n    [TABLES / "manuscript_calibration.csv"],\n)\n\n# ------------------------------------------------------------------\n# Figure 4 — Applicability domain / abstention\n# ------------------------------------------------------------------\nad_df = load_csv(\n    TABLES / "manuscript_ad.csv",\n    required=["endpoint", "applicability_domain", "n", "supported", "accuracy"],\n).copy()\n\nad_df["support_rate"] = np.where(\n    ad_df["n"] > 0,\n    ad_df["supported"] / ad_df["n"],\n    np.nan,\n)\n\ndomain_order = ["inside", "borderline", "outside"]\n\nfig, axes = plt.subplots(1, 2, figsize=(8.2, 4.0))\n\nsupport_pivot = (\n    ad_df.pivot_table(\n        index="endpoint",\n        columns="applicability_domain",\n        values="support_rate",\n        aggfunc="first",\n    )\n    .reindex([e for e in ENDPOINT_ORDER if e in ad_df["endpoint"].unique()])\n)\n\nx = np.arange(len(support_pivot.index))\nwidth = 0.24\n\nfor k, domain in enumerate(domain_order):\n    if domain in support_pivot.columns:\n        axes[0].bar(\n            x + (k - 1) * width,\n            100 * support_pivot[domain],\n            width,\n            label=domain.capitalize(),\n        )\n\naxes[0].set_xticks(x)\naxes[0].set_xticklabels(\n    [endpoint_label(e) for e in support_pivot.index],\n    rotation=35,\n    ha="right",\n)\naxes[0].set_ylabel("Supported predictions (%)")\naxes[0].set_title("A. Decision support by domain stratum")\naxes[0].legend(frameon=False)\n\nsupported_df = ad_df.loc[\n    (ad_df["supported"] > 0) &\n    (ad_df["applicability_domain"].isin(["inside", "borderline"]))\n].copy()\n\nacc_pivot = (\n    supported_df.pivot_table(\n        index="endpoint",\n        columns="applicability_domain",\n        values="accuracy",\n        aggfunc="first",\n    )\n    .reindex(support_pivot.index)\n)\n\nfor k, domain in enumerate(["inside", "borderline"]):\n    if domain in acc_pivot.columns:\n        axes[1].bar(\n            x + (k - 0.5) * 0.34,\n            100 * acc_pivot[domain],\n            0.34,\n            label=domain.capitalize(),\n        )\n\naxes[1].set_xticks(x)\naxes[1].set_xticklabels(\n    [endpoint_label(e) for e in acc_pivot.index],\n    rotation=35,\n    ha="right",\n)\naxes[1].set_ylabel("Accuracy (%)")\naxes[1].set_title("B. Accuracy in supported strata")\naxes[1].legend(frameon=False)\n\nfig.tight_layout()\n\nsave_pdf(\n    fig,\n    "Fig04_applicability_domain_abstention",\n    "Applicability-domain support rates and accuracy in supported strata.",\n    [TABLES / "manuscript_ad.csv"],\n)\n\n# ------------------------------------------------------------------\n# Figure 5 — Activity cliffs\n# ------------------------------------------------------------------\ncliff_summary = load_csv(\n    TABLES / "manuscript_cliffs.csv",\n    required=["endpoint", "cliff_pairs"],\n).copy()\n\ncliff_pairs = optional_csv(\n    RESULTS / "activity_cliffs" / "cliff_pairs.csv",\n    required=["endpoint", "tanimoto", "label_a", "label_b"],\n)\n\ncliff_summary["endpoint"] = pd.Categorical(\n    cliff_summary["endpoint"], categories=ENDPOINT_ORDER, ordered=True\n)\ncliff_summary = cliff_summary.sort_values("endpoint")\ncliff_summary["endpoint_label"] = cliff_summary["endpoint"].map(endpoint_label)\n\nfig, axes = plt.subplots(1, 2, figsize=(8.3, 4.0))\n\naxes[0].bar(cliff_summary["endpoint_label"], cliff_summary["cliff_pairs"])\naxes[0].set_ylabel("Activity-cliff pairs")\naxes[0].set_title("A. Activity-cliff burden")\naxes[0].tick_params(axis="x", rotation=35)\n\nif cliff_pairs is not None and not cliff_pairs.empty:\n    box_data = []\n    labels = []\n    for endpoint in ENDPOINT_ORDER:\n        values = cliff_pairs.loc[\n            cliff_pairs["endpoint"].eq(endpoint), "tanimoto"\n        ].dropna().to_numpy()\n        if len(values):\n            box_data.append(values)\n            labels.append(endpoint_label(endpoint))\n\n    axes[1].boxplot(box_data, tick_labels=labels, showfliers=False)\n    axes[1].set_ylabel("Tanimoto similarity")\n    axes[1].set_title("B. Similarity of cliff pairs")\n    axes[1].tick_params(axis="x", rotation=35)\nelse:\n    axes[1].text(0.5, 0.5, "cliff_pairs.csv unavailable", ha="center", va="center")\n    axes[1].set_axis_off()\n\nfig.tight_layout()\n\nsource_files = [TABLES / "manuscript_cliffs.csv"]\nif cliff_pairs is not None:\n    source_files.append(RESULTS / "activity_cliffs" / "cliff_pairs.csv")\n\nsave_pdf(\n    fig,\n    "Fig05_activity_cliffs",\n    "Activity-cliff counts and Tanimoto similarity distributions.",\n    source_files,\n)\n\n# ------------------------------------------------------------------\n# Figure 6 — Multitask transfer\n# ------------------------------------------------------------------\ntransfer_df = optional_csv(\n    RESULTS / "ablations" / "negative_transfer.csv",\n    required=["endpoint", "metric", "single_task", "multitask", "transfer", "category"],\n)\n\nif transfer_df is not None and not transfer_df.empty:\n    transfer_df = transfer_df.loc[transfer_df["metric"].eq("pr_auc")].copy()\n    transfer_df["endpoint"] = pd.Categorical(\n        transfer_df["endpoint"], categories=ENDPOINT_ORDER, ordered=True\n    )\n    transfer_df = transfer_df.sort_values("endpoint")\n    transfer_df["endpoint_label"] = transfer_df["endpoint"].map(endpoint_label)\n\n    fig, ax = plt.subplots(figsize=(6.5, 4.0))\n    y = np.arange(len(transfer_df))\n    ax.barh(y, transfer_df["transfer"])\n    ax.axvline(0, linewidth=1)\n    ax.set_yticks(y)\n    ax.set_yticklabels(transfer_df["endpoint_label"])\n    ax.set_xlabel("Delta PR-AUC (multitask - single-task)")\n    ax.set_title("Endpoint-specific multitask transfer")\n\n    for i, row in transfer_df.reset_index(drop=True).iterrows():\n        ax.text(\n            row["transfer"],\n            i,\n            f\' {row["transfer"]:+.3f} ({row["category"]})\',\n            va="center",\n            fontsize=7,\n        )\n\n    fig.tight_layout()\n    save_pdf(\n        fig,\n        "Fig06_multitask_transfer",\n        "Endpoint-specific change in PR-AUC from single-task to multitask learning.",\n        [RESULTS / "ablations" / "negative_transfer.csv"],\n    )\n\n# ------------------------------------------------------------------\n# Figure 7 — External validation\n# ------------------------------------------------------------------\nexternal_df = optional_csv(\n    RESULTS / "external_validation" / "external_metrics.csv",\n    required=[\n        "dataset_name", "endpoint", "subset", "coverage",\n        "n", "pr_auc", "abstention_rate"\n    ],\n)\n\nif external_df is not None and not external_df.empty:\n    ext_all = external_df.loc[\n        external_df["subset"].eq("all_strict_evaluable")\n    ].copy()\n\n    ext_supported = external_df.loc[\n        external_df["subset"].eq("decision_supported")\n    ].copy()\n\n    internal = cal_df[["endpoint", "test_pr_auc"]].rename(\n        columns={"test_pr_auc": "internal_pr_auc"}\n    )\n\n    paired = internal.merge(\n        ext_all[["endpoint", "pr_auc"]].rename(\n            columns={"pr_auc": "external_pr_auc"}\n        ),\n        on="endpoint",\n        how="inner",\n    )\n\n    paired["endpoint"] = pd.Categorical(\n        paired["endpoint"], categories=ENDPOINT_ORDER, ordered=True\n    )\n    paired = paired.sort_values("endpoint")\n\n    fig, axes = plt.subplots(1, 2, figsize=(8.1, 4.0))\n\n    for _, row in paired.iterrows():\n        axes[0].plot(\n            [0, 1],\n            [row["internal_pr_auc"], row["external_pr_auc"]],\n            marker="o",\n            label=endpoint_label(row["endpoint"]),\n        )\n\n    axes[0].set_xticks([0, 1])\n    axes[0].set_xticklabels(["Internal\\nscaffold test", "External\\nTox21 final"])\n    axes[0].set_ylabel("PR-AUC")\n    axes[0].set_title("A. Internal-to-external generalization")\n    axes[0].legend(frameon=False)\n\n    if not ext_supported.empty:\n        coverage_df = ext_supported.copy()\n        coverage_df["endpoint"] = pd.Categorical(\n            coverage_df["endpoint"], categories=ENDPOINT_ORDER, ordered=True\n        )\n        coverage_df = coverage_df.sort_values("endpoint")\n\n        axes[1].bar(\n            [endpoint_label(e) for e in coverage_df["endpoint"]],\n            100 * coverage_df["coverage"],\n        )\n        axes[1].set_ylabel("Decision-supported coverage (%)")\n        axes[1].set_title("B. External prediction coverage")\n        axes[1].tick_params(axis="x", rotation=35)\n\n    fig.tight_layout()\n\n    save_pdf(\n        fig,\n        "Fig07_external_validation",\n        "Internal scaffold-test versus external PR-AUC and supported coverage.",\n        [\n            TABLES / "manuscript_calibration.csv",\n            RESULTS / "external_validation" / "external_metrics.csv",\n        ],\n    )\n\n# ------------------------------------------------------------------\n# Supplementary Figure S3 — repeated scaffold stability\n# ------------------------------------------------------------------\nrepeat_df = optional_csv(\n    RESULTS / "metrics" / "qsar_repeated_scaffold_summary.csv",\n    required=[\n        "endpoint", "model", "metric", "runs",\n        "mean", "std", "ci95_lower", "ci95_upper"\n    ],\n)\n\nif repeat_df is not None and not repeat_df.empty:\n    repeat_pr = repeat_df.loc[repeat_df["metric"].eq("pr_auc")].copy()\n    repeat_pr = repeat_pr.loc[repeat_pr["endpoint"].isin(ENDPOINT_ORDER)].copy()\n\n    endpoints_present = [e for e in ENDPOINT_ORDER if e in repeat_pr["endpoint"].unique()]\n    models_present = (\n        repeat_pr.groupby("model")["mean"]\n        .mean()\n        .sort_values(ascending=False)\n        .index.tolist()\n    )\n\n    x = np.arange(len(endpoints_present), dtype=float)\n    offsets = (\n        np.linspace(-0.25, 0.25, len(models_present))\n        if len(models_present) > 1 else np.array([0.0])\n    )\n\n    fig, ax = plt.subplots(figsize=(7.4, 4.2))\n\n    for offset, model in zip(offsets, models_present):\n        rows = (\n            repeat_pr.loc[repeat_pr["model"].eq(model)]\n            .set_index("endpoint")\n            .reindex(endpoints_present)\n        )\n\n        means = rows["mean"].to_numpy(dtype=float)\n        lower = rows["ci95_lower"].to_numpy(dtype=float)\n        upper = rows["ci95_upper"].to_numpy(dtype=float)\n        yerr = np.vstack([means - lower, upper - means])\n\n        ax.errorbar(\n            x + offset,\n            means,\n            yerr=yerr,\n            marker="o",\n            linestyle="none",\n            capsize=3,\n            label=model_label(model),\n        )\n\n    ax.set_xticks(x)\n    ax.set_xticklabels(\n        [endpoint_label(e) for e in endpoints_present],\n        rotation=35,\n        ha="right",\n    )\n    ax.set_ylabel("Repeated scaffold-split PR-AUC")\n    ax.set_title("Five-run scaffold-split stability (mean and 95% CI)")\n    ax.legend(frameon=False, ncols=min(3, max(1, len(models_present))))\n    fig.tight_layout()\n\n    save_pdf(\n        fig,\n        "FigS03_repeated_scaffold_pr_auc",\n        "Repeated scaffold-split PR-AUC means and 95% confidence intervals.",\n        [RESULTS / "metrics" / "qsar_repeated_scaffold_summary.csv"],\n    )\n\nmanifest_df = pd.DataFrame(manifest)\nmanifest_path = FIG_DIR / "figure_manifest.csv"\nmanifest_df.to_csv(manifest_path, index=False)\n\npdf_files = sorted(FIG_DIR.glob("*.pdf"))\nfor path in pdf_files:\n    if path.stat().st_size == 0:\n        raise RuntimeError(f"Empty figure detected: {path}")\n\nprint(f"[DONE] generated={len(manifest_df)} figure PDFs")\nprint(f"[DONE] manifest={manifest_path}")\n'

with tempfile.NamedTemporaryFile(
    mode="w",
    suffix="_jcim_plotter.py",
    encoding="utf-8",
    delete=False,
) as tmp:
    tmp.write(plot_script)
    tmp_script = Path(tmp.name)

try:
    result = subprocess.run(
        [
            str(plot_python),
            "-X",
            "faulthandler",
            str(tmp_script),
        ],
        cwd=ROOT,
        capture_output=True,
        text=True,
        timeout=600,
    )

    print("Return code:", result.returncode)

    if result.stdout:
        print("\n--- plotting stdout ---")
        print(result.stdout)

    if result.stderr:
        print("\n--- plotting stderr ---")
        print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(
            "The separate plotting process failed. "
            "The Jupyter kernel is still alive; inspect stderr above."
        )
finally:
    try:
        tmp_script.unlink(missing_ok=True)
    except Exception:
        pass


Return code: 0

--- plotting stdout ---
[PLOT] created D:\Dropbox\Work\Learning\Python\toxicity_screening_project\figures\jcim_manuscript_pdf\Fig01_endpoint_data_landscape.pdf
[PLOT] created D:\Dropbox\Work\Learning\Python\toxicity_screening_project\figures\jcim_manuscript_pdf\Fig02_scaffold_test_pr_auc.pdf
[PLOT] created D:\Dropbox\Work\Learning\Python\toxicity_screening_project\figures\jcim_manuscript_pdf\FigS01_scaffold_test_mcc.pdf
[PLOT] created D:\Dropbox\Work\Learning\Python\toxicity_screening_project\figures\jcim_manuscript_pdf\FigS02_scaffold_test_brier.pdf
[PLOT] created D:\Dropbox\Work\Learning\Python\toxicity_screening_project\figures\jcim_manuscript_pdf\Fig03_reliability_tradeoff.pdf
[PLOT] created D:\Dropbox\Work\Learning\Python\toxicity_screening_project\figures\jcim_manuscript_pdf\Fig04_applicability_domain_abstention.pdf
[PLOT] created D:\Dropbox\Work\Learning\Python\toxicity_screening_project\figures\jcim_manuscript_pdf\Fig05_activity_cliffs.pdf
[PLOT] created D:\Drop

## Verify generated files

In [3]:

FIG_DIR = ROOT / "figures" / "jcim_manuscript_pdf"

pdfs = sorted(FIG_DIR.glob("*.pdf"))
print(f"PDF figures found: {len(pdfs)}")

for path in pdfs:
    print(f"{path.name:45s} {path.stat().st_size / 1024:9.1f} KB")

manifest = FIG_DIR / "figure_manifest.csv"
print("\nManifest exists:", manifest.exists())
print("Manifest path  :", manifest)


PDF figures found: 10
Fig01_endpoint_data_landscape.pdf                  15.0 KB
Fig02_scaffold_test_pr_auc.pdf                     41.3 KB
Fig03_reliability_tradeoff.pdf                     14.6 KB
Fig04_applicability_domain_abstention.pdf          13.8 KB
Fig05_activity_cliffs.pdf                          14.0 KB
Fig06_multitask_transfer.pdf                       13.3 KB
Fig07_external_validation.pdf                      16.0 KB
FigS01_scaffold_test_mcc.pdf                       40.3 KB
FigS02_scaffold_test_brier.pdf                     39.3 KB
FigS03_repeated_scaffold_pr_auc.pdf                15.7 KB

Manifest exists: True
Manifest path  : D:\Dropbox\Work\Learning\Python\toxicity_screening_project\figures\jcim_manuscript_pdf\figure_manifest.csv
